# S6 — AndinaLog 03B | Regresión térmica a 60 minutos

Se compara un baseline, modelos con variables base y los mismos modelos enriquecidos con eventos históricos. La selección usa validación; TEST se abre una vez.

## 1. Datos y objetivo

La unidad es una lectura IoT. El objetivo continuo es `max_desvio_termico_proximos_60min_c`; puede ser negativo cuando la ventana futura permanece dentro del umbral.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
SEMILLA = 42
HORIZONTE_MIN = 60
TARGET = "max_desvio_termico_proximos_60min_c"

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    else:
        raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró la raíz del proyecto")
    return raiz

RAIZ = encontrar_raiz()
RUTA_DATOS = RAIZ / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv"
SALIDAS = RAIZ / "proyecto-integrador/04_regresion/salidas_s6"
df = pd.read_csv(RUTA_DATOS, encoding="utf-8-sig")
df["timestamp_bolivia"] = pd.to_datetime(df["timestamp_bolivia"], errors="raise")
df["apta_regresion_60min"] = df["apta_regresion_60min"].astype("boolean")
modelado = df.loc[df["apta_regresion_60min"].fillna(False) & df[TARGET].notna()].copy()
assert len(df) == 28677 and len(modelado) == 27477
print("Lecturas totales:", len(df), "| aptas para regresión:", len(modelado))


Lecturas totales: 28677 | aptas para regresión: 27477


## 2. Variables disponibles y control de fuga

No se incluyen el objetivo binario, número de lecturas futuras, aptitud ni datos posteriores. Las variables de eventos usan solo historia anterior.

In [2]:
# Variables conocidas en el instante de la lectura.
NUM_BASE = [
    "temp_c", "humedad_pct", "objetivo_c", "tolerancia_c", "desvio_respecto_umbral_c",
    "temp_lag1_c", "temp_lag2_c", "cambio_temp_c", "pendiente_c_por_min",
    "temp_media_historica_3", "temp_max_historica_3", "minutos_desde_inicio",
    "capacidad_kg_tratada", "cantidad_solicitada_tratado", "tiempo_entrega_prometido_hrs_tratado",
]
CAT_BASE = ["categoria_logistica_tratada", "tipo_camion_tratado", "centro_distribucion_tratado"]
NUM_EVENTOS = [
    "eventos_previos_60m", "eventos_previos_180m", "eventos_previos_24h",
    "alertas_temp_previas_60m", "alertas_temp_previas_180m", "alertas_temp_previas_24h",
    "fallas_motor_previas_24h", "eventos_alta_previos_24h",
    "eventos_no_reconocidos_previos_24h", "reconocimiento_faltante_previos_24h",
    "minutos_desde_ultimo_evento", "minutos_desde_ultima_alerta_temp",
    "umbral_eventos_temp_cabina_c", "geocerca_eventos_radio_km",
    "dias_desde_ultimo_mantenimiento_eventos",
]
FEATURES_BASE = NUM_BASE + CAT_BASE
FEATURES_EVENTOS = NUM_BASE + NUM_EVENTOS + CAT_BASE
PROHIBIDAS = {TARGET, "clasificacion_objetivo_60min", "n_lecturas_futuras_60min",
              "apta_clasificacion_60min", "apta_regresion_60min"}
assert not (set(FEATURES_EVENTOS) & PROHIBIDAS)
faltantes = set(FEATURES_EVENTOS + [TARGET, "viaje_id", "timestamp_bolivia"]) - set(modelado.columns)
if faltantes: raise ValueError(f"Faltan columnas: {sorted(faltantes)}")
print("Variables base:", len(FEATURES_BASE), "| variables con eventos:", len(FEATURES_EVENTOS))


Variables base: 18 | variables con eventos: 33


## 3. Split temporal por viaje

Cada viaje pertenece a una sola partición. Se deja un margen de 60 minutos entre bloques y se guardan las asignaciones para S7–S10.

In [3]:
# Split temporal con viajes completos y margen equivalente al horizonte.
ventanas = modelado.groupby("viaje_id")["timestamp_bolivia"].agg(inicio="min", fin="max").sort_values("inicio")
if len(ventanas) < 10: raise ValueError("No hay suficientes viajes para tres particiones")

pos_test = min(len(ventanas)-1, max(1, int(np.floor(0.85 * len(ventanas)))))
corte_test = ventanas.iloc[pos_test]["inicio"]
pretest = ventanas.index[ventanas["fin"].add(pd.Timedelta(minutes=HORIZONTE_MIN)).lt(corte_test)]
test = ventanas.index[ventanas["inicio"].ge(corte_test)]
excl_test = ventanas.index.difference(pretest.union(test))

v_pre = ventanas.loc[pretest].sort_values("inicio")
pos_val = min(len(v_pre)-1, max(1, int(np.floor(0.82 * len(v_pre)))))
corte_val = v_pre.iloc[pos_val]["inicio"]
ajuste = v_pre.index[v_pre["fin"].add(pd.Timedelta(minutes=HORIZONTE_MIN)).lt(corte_val)]
validacion = v_pre.index[v_pre["inicio"].ge(corte_val)]
excl_val = v_pre.index.difference(ajuste.union(validacion))

asignacion = ventanas.reset_index()[["viaje_id", "inicio", "fin"]]
asignacion["particion"] = "EXCLUIDO_MARGEN"
asignacion.loc[asignacion["viaje_id"].isin(ajuste), "particion"] = "AJUSTE"
asignacion.loc[asignacion["viaje_id"].isin(validacion), "particion"] = "VALIDACION"
asignacion.loc[asignacion["viaje_id"].isin(test), "particion"] = "TEST"
asignacion["motivo"] = np.where(asignacion["particion"].eq("EXCLUIDO_MARGEN"),
                                 "Viaje cruza un corte o su margen futuro de 60 minutos", "")
particion = modelado["viaje_id"].map(asignacion.set_index("viaje_id")["particion"])
idx_aj = modelado.index[particion.eq("AJUSTE")]
idx_val = modelado.index[particion.eq("VALIDACION")]
idx_test = modelado.index[particion.eq("TEST")]
idx_pretest = modelado.index[modelado["viaje_id"].isin(pretest)]

assert set(ajuste).isdisjoint(validacion) and set(pretest).isdisjoint(test)
assert ventanas.loc[ajuste, "fin"].max() + pd.Timedelta(minutes=60) < ventanas.loc[validacion, "inicio"].min()
assert ventanas.loc[pretest, "fin"].max() + pd.Timedelta(minutes=60) < ventanas.loc[test, "inicio"].min()
assert min(len(idx_aj), len(idx_val), len(idx_test)) > 0
print("Cortes:", corte_val, corte_test)
print(asignacion["particion"].value_counts().to_string())
print("Lecturas ajuste/validación/test:", len(idx_aj), len(idx_val), len(idx_test))


Cortes: 2026-08-21 00:50:00 2026-08-26 09:03:00
particion
AJUSTE             796
VALIDACION         180
TEST               180
EXCLUIDO_MARGEN     44
Lecturas ajuste/validación/test: 18221 4129 4119


## 4. Baseline y selección en validación

Se comparan Dummy, regresión lineal, Ridge y Random Forest, cada modelo clásico con variables base y con eventos.

In [4]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def preprocesador(numericas, categoricas, escalar=True):
    pasos_num = [("imputar", SimpleImputer(strategy="median", add_indicator=True))]
    if escalar: pasos_num.append(("escalar", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(pasos_num), numericas),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
    ])

def crear_modelos():
    modelos = {"Dummy media": (FEATURES_BASE, DummyRegressor(strategy="mean"))}
    for etiqueta, nums, feats in [("Base", NUM_BASE, FEATURES_BASE),
                                  ("Con eventos", NUM_BASE + NUM_EVENTOS, FEATURES_EVENTOS)]:
        modelos[f"Lineal - {etiqueta}"] = (feats, Pipeline([
            ("pre", preprocesador(nums, CAT_BASE, True)), ("modelo", LinearRegression())]))
        modelos[f"Ridge - {etiqueta}"] = (feats, Pipeline([
            ("pre", preprocesador(nums, CAT_BASE, True)), ("modelo", Ridge(alpha=1.0))]))
        modelos[f"Random Forest - {etiqueta}"] = (feats, Pipeline([
            ("pre", preprocesador(nums, CAT_BASE, False)),
            ("modelo", RandomForestRegressor(n_estimators=150, max_depth=10, min_samples_leaf=5,
                                              random_state=SEMILLA, n_jobs=-1))]))
    return modelos

def metricas(real, pred):
    mse = mean_squared_error(real, pred)
    return {"MAE": mean_absolute_error(real, pred), "RMSE": np.sqrt(mse), "R2": r2_score(real, pred)}

modelos = crear_modelos()
y = modelado[TARGET].astype(float)
filas_val, pred_val = [], {}
for nombre, (feats, modelo) in modelos.items():
    modelo.fit(modelado.loc[idx_aj, feats], y.loc[idx_aj])
    pred = modelo.predict(modelado.loc[idx_val, feats])
    pred_val[nombre] = pred
    filas_val.append({"modelo": nombre, "familia_variables": "eventos" if "Con eventos" in nombre else "base",
                      "particion": "VALIDACION", **metricas(y.loc[idx_val], pred)})
metricas_val = pd.DataFrame(filas_val).sort_values("RMSE")
candidatos = metricas_val.loc[~metricas_val["modelo"].eq("Dummy media")]
seleccion = candidatos.iloc[0]["modelo"]
print(metricas_val.round(4).to_string(index=False))
print("Seleccionado antes de abrir TEST:", seleccion)


                     modelo familia_variables  particion    MAE   RMSE      R2
       Random Forest - Base              base VALIDACION 0.7032 0.9994  0.5668
               Ridge - Base              base VALIDACION 0.7686 1.0746  0.4992
              Lineal - Base              base VALIDACION 0.7686 1.0746  0.4992
        Ridge - Con eventos           eventos VALIDACION 0.7661 1.0757  0.4982
       Lineal - Con eventos           eventos VALIDACION 0.7661 1.0757  0.4981
Random Forest - Con eventos           eventos VALIDACION 0.8135 1.0951  0.4799
                Dummy media              base VALIDACION 1.1599 1.5190 -0.0007
Seleccionado antes de abrir TEST: Random Forest - Base


## 5. Test final sellado

Después de seleccionar por RMSE de validación, se reajusta con todo PRETEST y se consulta TEST una sola vez.

In [5]:
# Apertura única de TEST después de seleccionar por validación.
nombre_base_pareado = seleccion.replace("Con eventos", "Base")
nombres_test = ["Dummy media", nombre_base_pareado, seleccion]
nombres_test = list(dict.fromkeys(nombres_test))
filas_test, pred_test = [], {}
for nombre in nombres_test:
    feats, modelo = crear_modelos()[nombre]
    modelo.fit(modelado.loc[idx_pretest, feats], y.loc[idx_pretest])
    pred = modelo.predict(modelado.loc[idx_test, feats])
    pred_test[nombre] = pred
    filas_test.append({"modelo": nombre, "familia_variables": "eventos" if "Con eventos" in nombre else "base",
                       "particion": "TEST", **metricas(y.loc[idx_test], pred)})
metricas_test = pd.DataFrame(filas_test).sort_values("RMSE")
rmse_dummy = float(metricas_test.loc[metricas_test["modelo"].eq("Dummy media"), "RMSE"].iloc[0])
metricas_test["mejora_rmse_vs_dummy_pct"] = 100 * (rmse_dummy - metricas_test["RMSE"]) / rmse_dummy
print(metricas_test.round(4).to_string(index=False))

pred_sel = pred_test[seleccion]
real_test = y.loc[idx_test].to_numpy()
residuo = real_test - pred_sel
positivos = real_test > 0
resumen_error = {
    "modelo_seleccionado": seleccion,
    "sesgo_medio_test_c": float(residuo.mean()),
    "p95_error_absoluto_test_c": float(np.percentile(np.abs(residuo), 95)),
    "proporcion_subestimaciones": float((residuo > 0).mean()),
    "excursiones_reales_test": int(positivos.sum()),
    "mae_excursiones_test_c": float(np.abs(residuo[positivos]).mean()) if positivos.any() else np.nan,
    "predicciones_no_positivas_ante_excursion_pct": float(100*(pred_sel[positivos] <= 0).mean()) if positivos.any() else np.nan,
}
print(resumen_error)


              modelo familia_variables particion    MAE   RMSE      R2  mejora_rmse_vs_dummy_pct
Random Forest - Base              base      TEST 0.7388 1.1243  0.6071                   37.6134
         Dummy media              base      TEST 1.2556 1.8022 -0.0096                    0.0000
{'modelo_seleccionado': 'Random Forest - Base', 'sesgo_medio_test_c': -0.003183031868652711, 'p95_error_absoluto_test_c': 1.9943809340523253, 'proporcion_subestimaciones': 0.4146637533381889, 'excursiones_reales_test': 232, 'mae_excursiones_test_c': 2.430595149948067, 'predicciones_no_positivas_ante_excursion_pct': 56.896551724137936}


## 6. Evidencias y residuos

Las métricas globales se complementan con errores por categoría, tipo de camión, centro y presencia de eventos. Estos grupos sirven para interpretar, no para volver a seleccionar.

In [6]:
# Exportación de evidencia reproducible.
SALIDAS.mkdir(parents=True, exist_ok=True)
metricas = pd.concat([metricas_val, metricas_test], ignore_index=True)
metricas.to_csv(SALIDAS / "metricas_regresion_s6.csv", index=False, encoding="utf-8-sig")
asig_export = asignacion.copy()
asig_export["inicio"] = asig_export["inicio"].dt.strftime("%Y-%m-%d %H:%M:%S")
asig_export["fin"] = asig_export["fin"].dt.strftime("%Y-%m-%d %H:%M:%S")
asig_export.to_csv(SALIDAS / "asignacion_split_viajes.csv", index=False, encoding="utf-8-sig")
predicciones = modelado.loc[idx_test, ["fila_bronze", "viaje_id", "timestamp_bolivia", TARGET,
                                      "categoria_logistica_tratada", "tipo_camion_tratado",
                                      "centro_distribucion_tratado", "tiene_evento_previo_24h"]].copy()
predicciones["modelo_seleccionado"] = seleccion
predicciones["prediccion_max_desvio_60min_c"] = pred_sel
predicciones["residuo_c"] = predicciones[TARGET] - predicciones["prediccion_max_desvio_60min_c"]
predicciones["timestamp_bolivia"] = predicciones["timestamp_bolivia"].dt.strftime("%Y-%m-%d %H:%M:%S")
predicciones.to_csv(SALIDAS / "predicciones_test_regresion_s6.csv", index=False, encoding="utf-8-sig")

# Diagnóstico por grupos en TEST, sin usarlo para re-seleccionar el modelo.
def error_grupo(g):
    return pd.Series({"lecturas": len(g), "MAE": np.abs(g["residuo_c"]).mean(),
                      "RMSE": np.sqrt(np.mean(g["residuo_c"]**2)),
                      "sesgo": g["residuo_c"].mean()})
grupos = []
for campo in ["categoria_logistica_tratada", "tipo_camion_tratado", "centro_distribucion_tratado", "tiene_evento_previo_24h"]:
    tabla = predicciones.groupby(campo, dropna=False).apply(error_grupo, include_groups=False).reset_index()
    tabla.insert(0, "dimension", campo)
    tabla = tabla.rename(columns={campo: "grupo"})
    grupos.append(tabla)
errores_grupo = pd.concat(grupos, ignore_index=True)
errores_grupo.to_csv(SALIDAS / "errores_test_por_grupo_s6.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(real_test, pred_sel, alpha=.3, s=10)
lim = [min(real_test.min(), pred_sel.min()), max(real_test.max(), pred_sel.max())]
axes[0].plot(lim, lim, "--", color="firebrick")
axes[0].set(title="TEST: real vs predicho", xlabel="Real (°C)", ylabel="Predicho (°C)")
axes[1].scatter(pred_sel, residuo, alpha=.3, s=10)
axes[1].axhline(0, linestyle="--", color="firebrick")
axes[1].set(title="TEST: residuos", xlabel="Predicho (°C)", ylabel="Real − predicho (°C)")
plt.tight_layout()
plt.savefig(SALIDAS / "diagnostico_residuos_s6.png", dpi=150, bbox_inches="tight")
plt.close()
print("Salidas guardadas en", SALIDAS)


Salidas guardadas en c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\04_regresion\salidas_s6


## 7. Alcance

El resultado es retrospectivo sobre datos sintéticos. Una mejora en este corte no demuestra desempeño operativo; antes de producción se necesita validación prospectiva, umbral de acción y monitoreo.